# 04 — Huấn luyện business calendar baselines và HGB cho app

Notebook tái tạo calendar baselines, rồi train riêng **HGB business-level** cho weather
observations và explicit scenarios của app. Ngách nhỏ là Source category, không phải ZIP. Một listing có nhiều tag
vẫn chỉ có một chuỗi Recorded check-ins.

Chuẩn bị source bằng 01/02 và import PostgreSQL theo [hướng dẫn local](../docs/agents/local-development.md).
Notebook chỉ đọc database. Mọi artifact nằm trong run riêng; `PUBLISH_TO_APP=False`
giữ nguyên model đã cài. Calendar baseline không được dùng làm fallback khi HGB unavailable.

HGB mặc định dùng observed station weather; seasonal/custom weather là explicit scenarios. Quyết định sản phẩm
này không chứng minh HGB có accuracy tốt hơn calendar. So sánh validation/test giữ nguyên
cả hai baseline, kể cả khi baseline có điểm tốt hơn. Thí nghiệm 03/05 và báo cáo 06/07 dùng
bounded source-category cohorts; bước HGB ở đây dùng toàn bộ supported imported cohorts.

## Context & Methods

**Câu hỏi:** với business có đủ lịch sử, pattern month/weekday nào dự báo được Recorded
check-ins trong một giai đoạn sau dữ liệu đã import?

| Fit | Cửa sổ học | Đánh giá |
| --- | --- | --- |
| Validation | 2017–2019 | 2020, chọn giữa overall mean và month/weekday shrunk |
| Assessment | 2018–2020, xây lại cohort | 2021, giữ nguyên lựa chọn từ validation |
| Final | 2019–2021, xây lại cohort | Projection; không có nhãn tương lai để chứng minh accuracy |

### Key Assumptions

- Business phải có first recorded activity không muộn hơn đầu cửa sổ học và ít nhất
  20 check-ins **trong cửa sổ đó**. Không dùng hoạt động tương lai để chọn cohort.
- Mỗi fit dùng đủ calendar days trong imported coverage; ngày không có check-in là zero.
  Ngày ngoài coverage hoặc failed read không được biến thành zero.
- Prior của month/weekday dùng cùng canonical city/state/parent, với 30 business-days
  shrinkage, đúng code production. Source category chỉ lọc listing, không chia nhãn.
- Metro là cohort Philadelphia/PA, Nashville/TN, Tampa/FL, không phải toàn bộ thị trường.
- Tags, tọa độ, tên và status là snapshot. Check-ins không phải khách hàng, doanh thu hay
  xác suất mở quán thành công. Những năm backtest này đã được xem trước nên vẫn exploratory.

### 1. Setup

In [ ]:
import hashlib
import json
import math
from datetime import UTC, date, datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from sitesense import (
    business_forecast,
    business_hgb,
    business_hgb_training,
    business_observed,
    business_training,
    location_weather,
    repository,
)
from sitesense.models import Filters

ROOT = next(
    directory
    for directory in (Path.cwd(), *Path.cwd().parents)
    if (directory / "pyproject.toml").is_file()
)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

### 2. Config

Các tham số dưới đây chỉ điều khiển ví dụ projection và nơi lưu kết quả. Training luôn
đọc đủ ba canonical city cohorts và hai parent categories để giữ đúng production priors.
Chỉ bật publish sau khi kiểm tra bundle mới và import revision.

In [ ]:
CITY = "Philadelphia"
STATE = "PA"
PARENT_CATEGORY = "Restaurants"  # UI: Restaurant; Spa uses "Day Spas".
SOURCE_CATEGORY = "Pizza"  # None means all source categories within the parent.
PROJECT_START = date(2026, 12, 1)
PROJECT_END = date(2026, 12, 31)
TOP_N = 10
TRAIN_APP_HGB = True
PUBLISH_TO_APP = False
HGB_WEATHER_MODE = "Observed weather"  # Or "Seasonal weather" / "Custom weather".
HGB_WEATHER_SCENARIO = None  # Custom only: (minimum °C, maximum °C, rain mm/day).

OUTPUT_ROOT = ROOT / "data/processed/business_forecast_training"
RUN_DIR = OUTPUT_ROOT / datetime.now(UTC).strftime("run_%Y%m%dT%H%M%S%fZ")
ARTIFACT_DIR = RUN_DIR / "artifact"  # Calendar reference.
HGB_ARTIFACT_DIR = RUN_DIR / "hgb_artifact"
RUN_DIR.mkdir(parents=True, exist_ok=False)

## Data

Load bằng chính read-only loader của app, không dựng nhãn từ ZIP hay nhân chia theo số
listing hiện tại. Credentials được truyền qua environment, không hiển thị connection URL.

In [ ]:
IMPORT_REVISION = repository.get_import_revision()
training_data = business_training.load_training_data()
assert repository.get_import_revision() == IMPORT_REVISION, "Import changed while reading."
assert training_data.coverage_start <= date(2017, 1, 1)
assert training_data.coverage_end >= date(2021, 12, 31)
assert PROJECT_START > training_data.coverage_end, "Projection must follow imported activity."
assert PROJECT_END >= PROJECT_START

inventory = pd.DataFrame(
    {
        "City": business.city,
        "State": business.state,
        "Business ID": business.business_id,
        "First recorded activity": business.first_activity,
    }
    for business in training_data.businesses
)
display(inventory.groupby(["City", "State"]).agg(Businesses=("Business ID", "nunique")))
display(
    pd.DataFrame(
        [
            {
                "Coverage start": training_data.coverage_start,
                "Coverage end": training_data.coverage_end,
                "Sparse nonzero/recorded day rows": len(training_data.activity),
                "Target": business_forecast.TARGET,
            }
        ]
    )
)

### 3. Train, select on validation, then assess

`business_training.train` xây lại cohort cho từng cửa sổ, đánh giá cả hai model trên cùng
business-parent/calendar-day keys và chọn model bằng validation 2020 trước khi đọc điểm
assessment 2021. Không chọn lại model vì điểm test thuận lợi hơn.

In [ ]:
final_groups, model_manifest = business_training.train(training_data, IMPORT_REVISION)
assert repository.get_import_revision() == IMPORT_REVISION, "Import changed during training."
assert model_manifest["grain"] == "business_id_parent_day"
assert model_manifest["weather_features"] is False
assert len(final_groups) == model_manifest["final_businesses"]

metrics = pd.DataFrame(
    {"Period": period, "Model": name, **values}
    for period, key in (
        ("Validation 2020", "validation_metrics"),
        ("Assessment 2021", "assessment_metrics"),
    )
    for name, values in model_manifest[key].items()
)
selected_model = min(
    business_forecast.MODEL_NAMES,
    key=lambda name: (
        model_manifest["validation_metrics"][name]["poisson_deviance"],
        business_forecast.MODEL_NAMES.index(name),
    ),
)
assert selected_model == model_manifest["model_name"]
metrics.to_csv(RUN_DIR / "evaluation_metrics.csv", index=False)
display(metrics)

## Results

Poisson deviance và MAE dùng trọng số bằng nhau cho từng business-parent/calendar-day,
bao gồm covered zero days. Top-business agreement so sánh các business trong cùng exact
source-category scope; các scope có tag chồng lặp không phải các thị trường độc lập.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for axis, metric, label in zip(
    axes,
    ("poisson_deviance", "mae"),
    ("Mean Poisson deviance", "Mean absolute error (recorded count)"),
    strict=True,
):
    metrics.pivot(index="Model", columns="Period", values=metric).plot.bar(ax=axis, rot=0)
    axis.set(title=label, xlabel="", ylabel=label, ylim=(0, None))
fig.tight_layout()
fig.savefig(RUN_DIR / "calendar_evaluation.png", bbox_inches="tight")
plt.show()

members = pd.DataFrame(
    {
        "City": group.city,
        "State": group.state,
        "Parent": group.category,
        "Business ID": group.business_id,
    }
    for group in final_groups
)
display(
    members.groupby(["City", "State", "Parent"]).agg(
        business_parent_members=("Business ID", "size"),
        unique_business_ids=("Business ID", "nunique"),
    )
)

### 4. Write and verify the production-format artifact

Cả payload và manifest phải khớp code, lockfile và import revision hiện tại. Loader không
bỏ qua provenance để làm cho ví dụ chạy được. Artifact research HGB/GAM/CatBoost không
được ghi vào thư mục bundle này.

In [ ]:
business_training.write_bundle(ARTIFACT_DIR, final_groups, model_manifest)
verified_bundle = business_forecast.load_bundle(ARTIFACT_DIR, IMPORT_REVISION)
assert verified_bundle.manifest["model_name"] == selected_model
assert verified_bundle.groups == final_groups
assert verified_bundle.manifest["code_sha256"] == business_forecast.code_sha256()
assert verified_bundle.manifest["lock_sha256"] == business_forecast.lock_sha256()
display(
    pd.DataFrame(
        [
            {
                "Model": selected_model,
                "Business-parent members": len(final_groups),
                "Training start": model_manifest["training_start"],
                "Training end": model_manifest["training_end"],
                "Weather inputs": False,
                "Artifact": str(ARTIFACT_DIR.relative_to(ROOT)),
            }
        ]
    )
)

### 5. Calendar reference: existing businesses in one exact niche

Xếp hạng bằng **Projected check-ins** cộng trên toàn bộ ngày trong khoảng được chọn.
Không gộp theo ZIP, không dùng check-ins per 100 businesses, và không nhân dự báo với số
listing hiện tại. Business không thuộc supported cohort không nhận dự báo giả định.

Bảng này là calendar reference. Forecast weather của app dùng verified HGB ở bước tiếp theo.

In [ ]:
projections = business_forecast.project(
    verified_bundle, CITY, STATE, PARENT_CATEGORY, SOURCE_CATEGORY, PROJECT_START, PROJECT_END
)
projection_table = pd.DataFrame(
    {
        "Business ID": row.group.business_id,
        "Business": row.group.name,
        "Latitude": row.group.latitude,
        "Longitude": row.group.longitude,
        "Source categories": ", ".join(row.group.source_categories),
        "Projected check-ins": row.period_rate,
        "Projected calendar days": row.calendar_days,
    }
    for row in projections
)
if projection_table.empty:
    display(
        Markdown(
            "Không có business được hỗ trợ trong ngách này. Kết quả unavailable, không phải zero."
        )
    )
else:
    assert all(math.isfinite(row.period_rate) and row.period_rate >= 0 for row in projections)
    assert all(
        SOURCE_CATEGORY is None or SOURCE_CATEGORY in row.group.source_categories
        for row in projections
    )
    projection_table = projection_table.sort_values(
        ["Projected check-ins", "Business ID"], ascending=[False, True]
    ).reset_index(drop=True)
    display(projection_table.head(TOP_N))
projection_table.to_csv(RUN_DIR / "business_projections.csv", index=False)

### 6. Train HGB and use observed weather or an explicit scenario

`TRAIN_APP_HGB=True` fit tất cả supported business/parent cohorts của ba canonical cities,
trong các batch identity có giới hạn, dùng cùng forward windows và accepted ACIS policy.
Cả HGB và calendar baselines được chấm trên cùng accepted-weather held-out keys. Calendar
và weather training coverage khác nhau; chênh lệch điểm không cô lập incremental weather gain.

Seasonal weather là mean station/month của training 2019–2021; custom tuple là
`(tmin_c, tmax_c, precipitation_mm)`. Đây đều là scenarios, không phải live weather forecasts.
Nếu tắt training và không có artifact riêng trong run, trạng thái HGB là unavailable.

Mặc định dùng accepted observed station weather, đúng app. Mỗi business phải có đủ
tmin/tmax/rain cho **toàn bộ** khoảng projection; thiếu ngày giữ unavailable, không zero
và không tự chuyển sang seasonal. Ngày tương lai chưa có observations sẽ chưa có estimates.
Chọn `Seasonal weather` hoặc `Custom weather` để chạy scenario có chủ đích.

In [ ]:
hgb_bundle = None
hgb_status = "unavailable: training not requested"
if HGB_WEATHER_MODE not in ("Observed weather", "Seasonal weather", "Custom weather"):
    raise ValueError("Choose Observed weather, Seasonal weather or Custom weather.")
if HGB_WEATHER_MODE == "Custom weather":
    if HGB_WEATHER_SCENARIO is None:
        raise ValueError("Custom weather requires an explicit temperature/rain tuple.")
    business_hgb.validate_weather(HGB_WEATHER_SCENARIO)
if TRAIN_APP_HGB:
    hgb_manifest = business_hgb_training.train_from_database(HGB_ARTIFACT_DIR)
    hgb_bundle = business_hgb.load_bundle(HGB_ARTIFACT_DIR, IMPORT_REVISION)
    hgb_status = "verified"
    hgb_metrics = pd.DataFrame(hgb_manifest["evaluation_metrics"])
    hgb_metrics.to_csv(RUN_DIR / "hgb_evaluation_metrics.csv", index=False)
    display(hgb_metrics)
    catalog = repository.load_catalog(CITY, STATE, PARENT_CATEGORY)
    scoped = tuple(
        business
        for business in catalog.businesses
        if SOURCE_CATEGORY is None or SOURCE_CATEGORY in business.categories
    )
    supported_ids = tuple(business.business_id for business in scoped)
    daily_estimates = ()
    if HGB_WEATHER_MODE == "Observed weather":
        observations = location_weather.load_daily_station_weather(
            catalog.weather_cells, PROJECT_START, PROJECT_END
        )
        observed_rows = pd.DataFrame(
            (
                {"Station": station, "Date": day.isoformat(), **values}
                for station, days in sorted(observations.items())
                for day, values in sorted(days.items())
            ),
            columns=["Station", "Date", "tmin", "tmax", "pcpn"],
        )
        observed_rows.to_csv(RUN_DIR / "projection_observed_weather.csv", index=False)
        hgb_projections, daily_estimates = business_observed.project(
            hgb_bundle,
            Filters(
                CITY,
                STATE,
                PARENT_CATEGORY,
                PROJECT_START,
                PROJECT_END,
                source_category=SOURCE_CATEGORY,
            ),
            (PROJECT_START, PROJECT_END),
            supported_ids,
            {business.business_id: business.weather_cell_id for business in scoped},
            observations,
        )
    else:
        hgb_projections = business_hgb.project(
            hgb_bundle,
            CITY,
            STATE,
            PARENT_CATEGORY,
            SOURCE_CATEGORY,
            PROJECT_START,
            PROJECT_END,
            supported_ids,
            HGB_WEATHER_SCENARIO if HGB_WEATHER_MODE == "Custom weather" else None,
        )
    hgb_table = pd.DataFrame(
        (
            {
                "Business ID": row.group.business_id,
                "Business": row.group.name,
                "Projected check-ins": row.period_rate,
                "Projected calendar days": row.calendar_days,
            }
            for row in hgb_projections
        ),
        columns=["Business ID", "Business", "Projected check-ins", "Projected calendar days"],
    )
    display(Markdown(f"**Weather input:** {HGB_WEATHER_MODE}."))
    if not hgb_table.empty:
        hgb_table = hgb_table.sort_values(
            ["Projected check-ins", "Business ID"], ascending=[False, True]
        )
        display(hgb_table.head(TOP_N))
    else:
        display(Markdown("HGB unavailable cho scope/weather này; không phải zero activity."))
    hgb_table.to_csv(RUN_DIR / "hgb_business_projections.csv", index=False)
    pd.DataFrame(daily_estimates, columns=["Business ID", "Date", "Projected check-ins"]).to_csv(
        RUN_DIR / "hgb_daily_projections.csv", index=False
    )
display(Markdown(f"**HGB artifact:** {hgb_status}. Calendar is a reference, never a fallback."))

### 7. Publish HGB only when explicitly enabled

Mặc định giữ riêng cả calendar reference và HGB bundle. Nếu bật `PUBLISH_TO_APP`,
chỉ verified HGB bundle được copy vào `models/business_hgb/`. Không thay đổi database
hoặc âm thầm thay HGB unavailable bằng calendar baseline.

In [ ]:
assert repository.get_import_revision() == IMPORT_REVISION
if PUBLISH_TO_APP:
    if hgb_bundle is None:
        raise ValueError("A verified HGB bundle is required before publishing.")
    business_hgb.MODEL_DIR.mkdir(parents=True, exist_ok=True)
    for name in ("models.joblib", "manifest.json"):
        pending = business_hgb.MODEL_DIR / (name + ".tmp")
        pending.write_bytes((HGB_ARTIFACT_DIR / name).read_bytes())
        pending.replace(business_hgb.MODEL_DIR / name)
    business_hgb.load_bundle(business_hgb.MODEL_DIR, IMPORT_REVISION)
    display(Markdown("Đã publish verified HGB bundle cho ứng dụng local."))
else:
    display(Markdown("Các model được giữ trong run riêng; model app không bị ghi đè."))

### 8. Save reproducibility evidence

In [ ]:
summary = (
    f"Calendar model selected on validation: **{selected_model}**. "
    f"Final fit contains **{len(final_groups):,} business-parent members**, using 2019–2021. "
    f"HGB status: **{hgb_status}**. Evidence: `{RUN_DIR.relative_to(ROOT)}`."
)
(RUN_DIR / "report.md").write_text("# Individual-business calendar training\n\n" + summary + "\n")
files = [path for path in RUN_DIR.rglob("*") if path.is_file()]
run_manifest = {
    **verified_bundle.manifest,
    "artifact_directory": "artifact",
    "calendar_artifact_directory": "artifact",
    "hgb_artifact_directory": "hgb_artifact" if hgb_bundle is not None else None,
    "app_model_name": "hgb",
    "app_hgb_status": hgb_status,
    "hgb_weather_mode": HGB_WEATHER_MODE,
    "hgb_weather_scenario": (
        HGB_WEATHER_SCENARIO if HGB_WEATHER_MODE == "Custom weather" else None
    ),
    "notebook": "notebooks/04_full_history_forecast_training.ipynb",
    "notebook_sha256": hashlib.sha256(
        (ROOT / "notebooks/04_full_history_forecast_training.ipynb").read_bytes()
    ).hexdigest(),
    "scope": {
        "city": CITY,
        "state": STATE,
        "parent": PARENT_CATEGORY,
        "source_category": SOURCE_CATEGORY,
    },
    "projection_start": str(PROJECT_START),
    "projection_end": str(PROJECT_END),
    "published_to_app": PUBLISH_TO_APP,
    "files_sha256": {
        str(path.relative_to(RUN_DIR)): hashlib.sha256(path.read_bytes()).hexdigest()
        for path in sorted(files)
    },
}
assert repository.get_import_revision() == IMPORT_REVISION
(RUN_DIR / "manifest.json").write_text(json.dumps(run_manifest, indent=2, sort_keys=True) + "\n")

## Takeaways

Các kết quả được tạo từ business-level fits, không giữ lại số liệu ZIP cũ. HGB cho app
dùng observed weather hoặc explicit scenarios; calendar baselines vẫn cần được báo cáo trung thực. Một model
được sản phẩm sử dụng không đồng nghĩa model thắng accuracy. Cả hai họ model chỉ mô tả
existing historical businesses, không xác định ngách hay bất động sản tối ưu để mở quán.

In [ ]:
display(Markdown(summary))